## Emotion Recognition from Speech

### 1.Install Libraries

In [ ]:
!pip install librosa soundfile kaggle

### 2.Upload Kaggle File

In [ ]:
from google.colab import files
files.upload()

### 3.Setup Kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### 4.Download Dataset

In [ ]:
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio

### 5.Extract Dataset

In [ ]:
import zipfile

zip_path = "/content/ravdess-emotional-speech-audio.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/ravdess")

### 6.Import Libraries

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

### 7.Emotion Labels

In [ ]:
emotion_dict = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful'
}

### 8.Data Augmentation Functions

In [ ]:
def noise(data):
    noise_amp = 0.035 * np.random.uniform() * np.amax(data)
    data = data + noise_amp * np.random.normal(size=data.shape[0])
    return data

def pitch(data, sample_rate, pitch_factor=0.7):
    return librosa.effects.pitch_shift(
        data,
        sr=sample_rate,
        n_steps=pitch_factor
    )

### 9.Feature Extraction

In [ ]:
def extract_features(file_path):

    data, sample_rate = librosa.load(
        file_path,
        duration=3,
        offset=0.5
    )

    result = []

    # Original Audio
    mfcc = np.mean(
        librosa.feature.mfcc(
            y=data,
            sr=sample_rate,
            n_mfcc=40
        ).T,
        axis=0
    )

    result.append(mfcc)

    # Noise Audio
    noise_data = noise(data)

    mfcc_noise = np.mean(
        librosa.feature.mfcc(
            y=noise_data,
            sr=sample_rate,
            n_mfcc=40
        ).T,
        axis=0
    )

    result.append(mfcc_noise)

    # Pitch Audio
    pitch_data = pitch(data, sample_rate)

    mfcc_pitch = np.mean(
        librosa.feature.mfcc(
            y=pitch_data,
            sr=sample_rate,
            n_mfcc=40
        ).T,
        axis=0
    )

    result.append(mfcc_pitch)

    return result

### 10.Load Dataset

In [ ]:
X = []
y = []

dataset_path = "/content/ravdess"

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.endswith(".wav"):

            file_path = os.path.join(root, file)

            emotion_code = file.split("-")[2]

            if emotion_code in emotion_dict:

                emotion = emotion_dict[emotion_code]

                features = extract_features(file_path)

                for feature in features:
                    X.append(feature)
                    y.append(emotion)

### 11.Convert Data

In [ ]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

### 12.Encode Labels

In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

y_categorical = to_categorical(y_encoded)

### 13.Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42
)

### 14.Reshape Data for LSTM

In [ ]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

### 15.Build LSTM Model

In [ ]:
model = Sequential()

model.add(
    LSTM(
        128,
        input_shape=(40,1),
        return_sequences=False
    )
)

model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))

model.add(Dropout(0.3))

model.add(Dense(y_categorical.shape[1], activation='softmax'))

### 16.Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### 17.Train Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)

### 18.Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

### 19.Predict Emotion

In [ ]:
predictions = model.predict(X_test)

predicted_label = np.argmax(predictions[0])

predicted_emotion = label_encoder.inverse_transform(
    [predicted_label]
)

print("Predicted Emotion:", predicted_emotion[0])

### 20.Save Model

In [ ]:
model.save("emotion_recognition_model.h5")

### 21.Accuracy Graph

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title('Model Accuracy')

plt.ylabel('Accuracy')

plt.xlabel('Epoch')

plt.legend(['Train', 'Validation'])

plt.show()